# Vela.jl vs nltiming/Discovery on a simulated ELL1 pulsar

Three independent posteriors on the same par/tim and the same white-noise model (EFAC = 1, no red noise):

1. Native Vela.jl (`SPNTA.lnpost` + emcee)
2. nltiming / Discovery with JUG + NUTS
3. nltiming / Discovery with the Vela host callback + PTMCMCSampler

The third path is the derivative-free route: Vela has no JAX derivatives, so `discovery_signals()` evaluates the delay through `jax.pure_callback` and PTMCMC walks the compiled Discovery `logL`. Overlay all three corners.

The dataset is a simplified J1909-3744 (`BINARY ELL1`, 100 barycentric TOAs, 10 years). Only `A1`, `TASC`, `EPS1`, and `EPS2` are free. Spin, sky, DM, and `PB` are frozen. The files are already a dual-engine MetaPulsar-style standalone `FORMAT 1` pair (PINT vs tempo2 residual-difference is checked in a private test, not here).

Vela’s extra phase offset `PHOFF` is analytically marginalized so every run samples the same four axes. Short pedagogical chains — scale them for science. Run from `examples/notebooks/`.


In [ ]:
import os
import sys
os.environ.setdefault("JAX_ENABLE_X64", "1")

import jax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import corner
import emcee
import discovery as ds
from pathlib import Path
from metapulsar import create_metapulsar
from nltiming import TimingSpec, load_run
import nltiming.sampling as nlts
from pyvela import SPNTA

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="WARNING")

nlts.numpyro.ensure_x64()


## A TimingPulsar

PINT-native host. Vela reads these files directly; nltiming attaches JUG or Vela to the same MetaPulsar object.


In [ ]:
DATA = Path("..") / "data" / "J1909-3744-sim"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1909-3744.par",
        "tim": DATA / "J1909-3744.tim",
        "timing_package": "pint",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
print(pulsar.name, len(pulsar.toas))


## Vela.jl

Native Vela posterior via `SPNTA.lnpost` and `emcee`. `PHOFF` is marginalized so the sampled names match nltiming. `rescale_samples` returns TASC as days from Vela’s epoch offset; adding that offset puts TASC on the same MJD axis nltiming uses.


In [ ]:
spnta = SPNTA(
    str(DATA / "J1909-3744.par"),
    str(DATA / "J1909-3744.tim"),
    analytic_marginalized_params=["PHOFF"],
)
print(list(spnta.param_names))

nwalkers = 4 * spnta.ndim
rng = np.random.default_rng(0)
p0 = np.array([spnta.prior_transform(rng.random(spnta.ndim)) for _ in range(nwalkers)])
sampler = emcee.EnsembleSampler(
    nwalkers, spnta.ndim, spnta.lnpost_vectorized, vectorize=True,
)
sampler.run_mcmc(p0, 4000, progress=True)
vela_epoch = np.asarray(spnta.param_offsets, dtype=float) / np.asarray(
    spnta.scale_factors, dtype=float
)
samples_vela = (
    spnta.rescale_samples(sampler.get_chain(flat=True, discard=2000, thin=4))
    + vela_epoch
)


## nltiming + Discovery (JUG / NUTS)

JUG engine, default inference (the four ELL1 axes are sampled; the phase offset is marginalized). Fixed EFAC = 1. This is the gradient path — do not pass a Vela context to `decentered_model` / `nuts`.


In [ ]:
spec = TimingSpec(engines={"pint": "jug"}, name="timing")
timing = spec.for_pulsar(pulsar)
print("sampled:", timing.sampled)
print("marginalized:", timing.marginalized)

nd = {f"{pulsar.name}_efac": 1.0}
likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, nd, add_equad=False),
    *timing.discovery_signals(),
])
model = nlts.numpyro.decentered_model(likelihood, timing, fixed=nd)

from numpyro.infer import init_to_value
mcmc = nlts.numpyro.nuts(
    model, timing,
    num_warmup=500, num_samples=4000, num_chains=1,
    init_strategy=init_to_value(values=nlts.numpyro.decentered_init_values(timing, model.transport)),
)
mcmc.run(jax.random.PRNGKey(0))
post = nlts.numpyro.posterior(mcmc, timing)


## nltiming + Discovery (Vela / PTMCMC)

Same pulsar and inference plan, Vela engine. Discovery's GP/Woodbury `logL` stays JIT-compiled; the timing delay is a value-only `jax.pure_callback`. `discovery_target` maps the PTMCMC vector (`q` in sampling coordinates; `q = 0` is the engine expansion) to engine-native **delta**. Write the run sidecar **before** sampling so `load_run` can decode the chain.


In [ ]:
import tempfile

spec_vela = TimingSpec(engines={"pint": "vela"}, name="timing")
timing_vela = spec_vela.for_pulsar(pulsar)
print("sampled:", timing_vela.sampled)
print("engine:", type(timing_vela.engine).__name__)

likelihood_vela = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, nd, add_equad=False),
    *timing_vela.discovery_signals(),
])
target = nlts.ptmcmc.discovery_target(likelihood_vela, timing_vela, fixed=nd)

outdir = Path(tempfile.mkdtemp(prefix="nlt_vela_disc_"))
timing_vela.write(
    outdir,
    likelihood="discovery",
    sampler="ptmcmc",
    chain_layout=target.chain_layout(),
)
sampler = nlts.ptmcmc.discovery_sampler(target, outdir=outdir)
# First logL call JITs the Discovery kernel; later steps pay one Vela callback.
sampler.sample(target.initial_point(), Niter=20_000)

run = load_run(outdir)
post_vela_disc = run.posterior(burn=0.25)
print("decoded:", list(post_vela_disc))

## Overlay

nltiming names carry the PTA suffix (`A1_combined`, …). Native Vela uses PINT names. Same order, same physical units (TASC in MJD). All three runs.


In [ ]:
names = list(timing.sampled)
vela_names = [n.removesuffix("_combined") for n in names]
idx = [list(spnta.param_names).index(n) for n in vela_names]
truth = timing.space.to_physical(np.zeros(len(names)), units="display")
truths = [float(np.asarray(truth[n]).reshape(-1)[0]) for n in names]
vela = samples_vela[:, idx]
disc = np.column_stack([
    np.asarray(post.posterior[k]).reshape(-1) for k in names
])
disc_vela = np.column_stack([
    np.asarray(post_vela_disc[k]).reshape(-1) for k in names
])
# density=True + 1/N weights: 1D panels share a unit-area scale despite N differing.
hist_kw = {"density": True, "histtype": "step"}

fig = corner.corner(
    vela, labels=names, color="C0", truths=truths,
    weights=np.full(len(vela), 1.0 / len(vela)),
    hist_kwargs={**hist_kw, "color": "C0"},
)
corner.corner(
    disc,
    fig=fig,
    color="C1",
    truths=truths,
    plot_datapoints=False,
    plot_density=False,
    fill_contours=False,
    no_fill_contours=True,
    weights=np.full(len(disc), 1.0 / len(disc)),
    hist_kwargs={**hist_kw, "color": "C1"},
)
corner.corner(
    disc_vela,
    fig=fig,
    color="C2",
    truths=truths,
    plot_datapoints=False,
    plot_density=False,
    fill_contours=False,
    no_fill_contours=True,
    weights=np.full(len(disc_vela), 1.0 / len(disc_vela)),
    hist_kwargs={**hist_kw, "color": "C2"},
)
fig.legend(
    handles=[
        Line2D([0], [0], color="C0", label="Vela.jl / emcee"),
        Line2D([0], [0], color="C1", label="nltiming / Discovery / JUG / NUTS"),
        Line2D([0], [0], color="C2", label="nltiming / Discovery / Vela / PTMCMC"),
    ],
    loc="upper right",
)
plt.show()
